# Direct questioning — DeepSeek-V3 on StrategyClaim (500 claims)

One-word true/false answer at temperature 0, without facts (68.6) and with the annotators' facts (80.2, reported in Table 1). The first cell loads the data; the second runs both variants.

In [ ]:
!pip install -q openai datasets scikit-learn

import os, re, sys, time, pickle
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

key = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
client = OpenAI(api_key=key, base_url="https://api.deepseek.com",
                timeout=60, max_retries=2)
MODEL_NAME = "deepseek-chat"

WORK_DIR = Path("/kaggle/working/baselines_deepseek")
WORK_DIR.mkdir(parents=True, exist_ok=True)

repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q "
              "https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk(repo + "/Datasets/StrategyQA/Experiment")
df = df.to_pandas().reset_index(drop=True)


def format_facts(f):
    if isinstance(f, (list, tuple, np.ndarray)):
        return "\n".join("- " + str(x) for x in f)
    return "" if f is None else str(f)


claims = df["claim"].astype(str).tolist()
facts = df["facts"].apply(format_facts).tolist()
gold = df["valid"].apply(lambda v: int(bool(v))).tolist()

sys.path.insert(0, repo)
import Uncertainpy.src.uncertainpy.gradual as grad
agg_f = grad.semantics.modular.ProductAggregation()
inf_f = grad.semantics.modular.LinearInfluence(conservativeness=1)

print(MODEL_NAME, "| claims:", len(claims), "| true:", sum(gold))
print("first facts:\n", facts[0])

deepseek-chat | claims: 500 | true: 250
first facts:
 ['Reza Shah, the founder of the Imperial state of Iran, was a Twelver Shia Muslim.'
 'Twelver Shia Muslims believe that there are 12 Imams.'
 'Jesus is typically represented as having 12 disciples.']


In [ ]:
import pickle, time
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)


def parse_tf(text):
    t = (text or "").strip().lower().lstrip("*\"' ")
    if t.startswith("true"):
        return 1
    if t.startswith("false"):
        return 0
    has_t, has_f = "true" in t, "false" in t
    if has_t and not has_f:
        return 1
    if has_f and not has_t:
        return 0
    return None


def ask(claim, fact=None):
    if fact:
        p = (f"Facts:\n{fact}\n\nUsing these facts, is the following "
             'statement true or false? Answer with only one word: '
             f'"true" or "false".\n\nStatement: {claim}\n\nAnswer:')
    else:
        p = ('Is the following statement true or false? Answer with only '
             f'one word: "true" or "false".\n\nStatement: {claim}\n\nAnswer:')
    for a in range(1, 9):
        try:
            r = client.chat.completions.create(
                model=MODEL_NAME, temperature=0, max_tokens=10,
                messages=[{"role": "user", "content": p}])
            return parse_tf(r.choices[0].message.content)
        except Exception:
            time.sleep(5 * a)
    return None


def report(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
    return {"method": name, "n": len(pred),
            "accuracy": round(accuracy_score(gold, pred) * 100, 1),
            "precision": round(precision_score(gold, pred,
                                               zero_division=0) * 100, 1),
            "recall": round(recall_score(gold, pred,
                                         zero_division=0) * 100, 1),
            "f1": round(f1_score(gold, pred, zero_division=0) * 100, 1),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}


DCK = WORK_DIR / "direct_strict_ckpt.pkl"
dres = pickle.load(open(DCK, "rb")) if DCK.exists() else {}
print("already done:", len(dres))

for i in range(len(claims)):
    if i in dres:
        continue
    dres[i] = {"no_facts": ask(claims[i]),
               "with_facts": ask(claims[i], facts[i])}
    pickle.dump(dres, open(DCK, "wb"))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(claims)}")

rows = []
for k in ["no_facts", "with_facts"]:
    pred = [dres[i][k] if dres[i][k] is not None else 0
            for i in range(len(claims))]
    fails = sum(dres[i][k] is None for i in range(len(claims)))
    row = report(f"Direct ({k})", pred)
    row["unparsed"] = fails
    rows.append(row)

print(f"\n=== {MODEL_NAME} - StrategyClaim - Direct ===")
print(pd.DataFrame(rows).to_string())

already done: 0
  50/500
  100/500
  150/500
  200/500
  250/500
  300/500
  350/500
  400/500
  450/500
  500/500

=== deepseek-chat - StrategyClaim - Direct ===
                method    n  accuracy  precision  recall    f1   TN  FP   FN   TP  unparsed
0    Direct (no_facts)  500      68.6       86.6    44.0  58.4  233  17  140  110         0
1  Direct (with_facts)  500      80.2       98.1    61.6  75.7  247   3   96  154         0
